                                               Objective 
This notebook will involve feature engineering the variables  so as the model will understand better the underline patterns and make predictions on anomalies more reliable

                                Feature Engineering 

This is a process of selecting, transforming and extract relevant features from raw data to enhamce a model accuracy and performance. This notebook will involve:
1. Selecting features
2. Transfroming categorical features into dummy variables, segemneting continous features like customer age into various groups.
3. Extract new features from the current features like time_difference, Transaction velocity, debit-balance-ratio, load-and-go that will make our model robust in anomaly detection 

finally saving the dataset into a featureengineered.csv

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import  StandardScaler 


In [ ]:
# uploading our dataset
df = pd.read_csv(r'./bank_transactions_data_2.csv')
df.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


In [208]:
df.shape

(2512, 16)

                   Check null values 
Handling null values by remove or modify them is crucial since they reduce a model predictive power, introduce biase since they make the dataset not to be representative of the population. Our dataset doesn't have null values.

In [209]:
df.isnull().sum().sum()

np.int64(0)

In [210]:
df.describe()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance
count,2512.000000,2512.000000,2512.000000,2512.000000,2512.000000
mean,297.593778,44.673965,119.643312,1.124602,5114.302966
std,291.946243,17.792198,69.963757,0.602662,3900.942499
min,0.260000,18.000000,10.000000,1.000000,101.250000
25%,81.885000,27.000000,63.000000,1.000000,1504.370000
50%,211.140000,45.000000,112.500000,1.000000,4735.510000
75%,414.527500,59.000000,161.000000,1.000000,7678.820000
max,1919.110000,80.000000,300.000000,5.000000,14977.990000


In [211]:
df.columns

Index(['TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate',
       'TransactionType', 'Location', 'DeviceID', 'IP Address', 'MerchantID',
       'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration',
       'LoginAttempts', 'AccountBalance', 'PreviousTransactionDate'],
      dtype='object')

In [212]:
df.dtypes

TransactionID               object
AccountID                   object
TransactionAmount          float64
TransactionDate             object
TransactionType             object
Location                    object
DeviceID                    object
IP Address                  object
MerchantID                  object
Channel                     object
CustomerAge                  int64
CustomerOccupation          object
TransactionDuration          int64
LoginAttempts                int64
AccountBalance             float64
PreviousTransactionDate     object
dtype: object

                                  Time change in minutes feature                                    

Converting transaction date from  object datatype into datetime to enable gettimg the difference in minutes an account makes a transaction. This will aid the model to flag those transactions with bot-like behaviour of debiting or crediting the account within few minutes interval. 

The difference method results in null valuesbin initial values thus we will assign resultant null values zeros.

In [213]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['TimeChange_mins'] = df.groupby('AccountID')['TransactionDate'].diff().dt.total_seconds()/ 60 
df['TimeChange_mins'] = df['TimeChange_mins'].fillna(0)




                                          Transaction velocity
This refers to the speed and frequency of transactions within a certain period. We will flag those transactions that occur within that minutes by leveraging t their frequency. This will help the model to label a transaction an anomaly if an account has a different transaction velocity comapred to the norm.

In [214]:
df = df.set_index('TransactionDate') 
df = df.sort_values(by=['AccountID', 'TransactionDate'])
df['TransactionVelocity_30mins'] = df.groupby('AccountID')['TransactionAmount'].rolling('30min').count().reset_index(level=0, drop=True)
df = df.reset_index()

In [215]:
df['CustomerAge'].describe()

count    2512.000000
mean       44.673965
std        17.792198
min        18.000000
25%        27.000000
50%        45.000000
75%        59.000000
max        80.000000
Name: CustomerAge, dtype: float64

                                       Customer age group

Binning customers age from a continous variable into groups of Genz, Millenial, Genx and Seniour to act a behaviour baselines thus reducing noise on the model tree splits thus shorter path lengths for anomalies thus increasing its efficiency. The segmentation criteria  is based on minimum age values of 18 being reprented in 0-20 bin while maximum customer age 80 to be represented by 55-80 bin


In [216]:
df['Customer_age_group'] = pd.cut(df['CustomerAge'], bins=[0, 20, 36, 55, 80] ,labels = ['GenZ', 'Millenial', 'GenX', 'Seniour'])


High_amount_for_job feature will flag those transactions in realtion to an ocuupation with amounts greater than 95%. This will aid the model to detect out of character transactions thus flag them as anomalies.

In [217]:
occupation_threshold = df.groupby('CustomerOccupation')['TransactionAmount'].transform(lambda x: x.quantile(0.95))

df['High_amount_for_job'] = (df['TransactionAmount'] > occupation_threshold).astype(int)


                               Debit-balance-ratio
Debit balance ratio will track how high a transaction is in relation to the account balance especially if a transaction is a debit. This feature will flag those transactions with a high ratio close to 1 which means an account is clearing his/ her bank account. Furthermore, it will distinguish those routinely high transaction amounts from wealthy individuals since they will have a low or normal debit-balance-ratio. 

Here, I assign credit transactions zero to only track transactions that are emptying the account.

NB: <bold>Adding 0.01 on account balance to avoid errors resulting from dividing transaction amount by zero</bold>

In [ ]:
df['Debit_Balance_ratio'] = np.where(df['TransactionType'] == 'Debit', (df['TransactionAmount'] / df['AccountBalance'] + 0.01), 0)


                                    Load-and-go feature
This involves crediting an account with money and debit/ or use it depending ion customer needs. To detect anomalies this feature will be created if:

1. A credit transaction amount is greater than 90 percentile happens
2. The account made a high amount debit that happenned  immediately  prior which was past the 90% threshold

if the  previoustransaction was a high value credit, the current transaction is a debit and transactions happened within 60 minutes, then it is flagged as a load and go transaction.

In [ ]:
credit_90th =  df[df['TransactionAmount'] == 'Credit']['TransactionAmount'].quantile(0.90)
df['is_high_credit'] = (df['TransactionType'] == 'Credit') & (df['TransactionAmount'] > credit_90th)

df['Prev_is_high_credit'] = df.groupby('AccountID')['is_high_credit'].shift(1).fillna(False).astype(bool)
df['Load_and_go_flag'] = ((df['Prev_is_high_credit'] == True) & (df['TransactionType']== 'Debit') & (df['TimeChange_mins'] <= 60)).astype(int)


C:\Users\test\AppData\Local\Temp\ipykernel_7588\998143002.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Prev_is_high_credit'] = df.groupby('AccountID')['is_high_credit'].shift(1).fillna(False)


In [220]:
df['Customer_age_group'].value_counts()

Customer_age_group
Millenial    826
Seniour      816
GenX         700
GenZ         170
Name: count, dtype: int64

                             Frequency encoding 
Features that are not ordinal(order doesn't matter) like TransactionType, Channel, CustomerOccupation will be encoded into numerical values based on their frequency in the dataset thus capturing the prevalence of a category helping in efficieny or correlation in the target variable identification.df_cleaned will be the resultant dataframe that will involve selected features, tranformed and extracted features.

NB: suffixing the TransactionType, channel and customeroccupation with 'freq' to differentiate encoded ones with the original ones.

In [221]:
for col in ['TransactionType', 'Channel', 'CustomerOccupation']:
    freq_map = df[col].value_counts(normalize=True) 
    df[col + '_freq'] = df[col].map(freq_map)
age_map = {'GenZ': 0, 'GenX': 1, 'Millenial': 2, 'Seniour': 3}
df['CustomerAge_map'] = df['Customer_age_group'].map(age_map)
df_cleaned = df[['TransactionID','TransactionAmount', 'TimeChange_mins', 'AccountBalance',  'Load_and_go_flag', 'Debit_Balance_ratio', 'High_amount_for_job', 'TransactionVelocity_30mins', 'TransactionType_freq', 'Channel_freq', 'CustomerOccupation_freq', 'CustomerAge_map']]
df_cleaned.head()

,TransactionID,TransactionAmount,TimeChange_mins,AccountBalance,Load_and_go_flag,Debit_Balance_ratio,High_amount_for_job,TransactionVelocity_30mins,TransactionType_freq,Channel_freq,CustomerOccupation_freq,CustomerAge_map
0,TX001313,47.79,0.000000,1649.92,0,0.028965,0,1.0,0.773885,0.345541,0.261545,2
1,TX002017,212.97,86396.233333,4180.40,0,0.050945,0,1.0,0.773885,0.322850,0.248806,3
2,TX002121,476.99,-351472.450000,1154.48,0,0.413164,0,1.0,0.773885,0.322850,0.261545,2
3,TX000021,59.32,0.000000,5750.89,0,0.010315,0,1.0,0.773885,0.345541,0.238455,3
4,TX001477,12.62,-331225.100000,6420.47,0,0.001966,0,1.0,0.773885,0.345541,0.251194,2


saving the resultant 'df_cleaned' dataframe into a dataset called bankfeature_engineered csv that will be fed into our model

In [222]:
df_cleaned.to_csv('./bankfeature_engineered.csv')
